# 🎬 PeakCut Studio: One-Click Google Colab Runner
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeharsh-patil/PeakCut/blob/main/PeakCut_Colab.ipynb)

**Turn long videos, podcasts, and live streams into viral 9:16 vertical shorts** using free NVIDIA GPU acceleration.

### ⚡ Quick Start:
1. Select GPU Runtime: **Runtime** → **Change runtime type** → select **T4 GPU**.
2. (Optional) Enter your **Gemini API Key(s)** below. Multiple comma-separated keys will automatically rotate when rate limits occur. If left blank, PeakCut's heuristic Virality Engine runs with zero API keys required!
3. Click the **Run button (▶)** on the code cell below.
4. Click the generated **`https://*.trycloudflare.com`** link to launch PeakCut Studio in your browser!

In [ ]:
# @title 🚀 Launch PeakCut Studio (One-Click Start)
# @markdown ### 🔑 Gemini API Keys (Optional, Comma-Separated)
# @markdown Enter one or multiple Gemini API keys. Multiple keys automatically rotate if one encounters quota/rate limits. Leave blank to run zero-cost heuristic virality mode.
GEMINI_API_KEYS = "" # @param {type:"string"}

# @markdown ### 🎙️ Speech-to-Text Model
WHISPER_MODEL = "base" # @param ["tiny", "base", "small", "medium", "large-v3"]

# @markdown ### ⚙️ Background Mode
# @markdown Check to run the server in the background (cell finishes with ✅ checkmark). Uncheck (default) to keep cell active and stream live video logs.
RUN_IN_BACKGROUND = False # @param {type:"boolean"}

# @markdown ### 📁 Save Directly to Google Drive (Persistent Storage)
# @markdown Check to mount Google Drive and automatically save all output clips into `MyDrive/PeakCut_Shorts`.
SAVE_TO_DRIVE = False # @param {type:"boolean"}

import os, sys, subprocess, time, re

# 0. Terminate lingering background processes from previous runs
subprocess.run("pkill -9 -f cloudflared; pkill -9 -f uvicorn", shell=True)

# Configure Environment Variables
if GEMINI_API_KEYS.strip():
    os.environ["GEMINI_API_KEYS"] = GEMINI_API_KEYS.strip()
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEYS.strip()

os.environ["WHISPER_MODEL"] = WHISPER_MODEL

# Auto-detect CUDA hardware acceleration
try:
    import torch
    has_cuda = torch.cuda.is_available()
except Exception:
    has_cuda = False

if has_cuda:
    os.environ["WHISPER_DEVICE"] = "cuda"
    os.environ["WHISPER_COMPUTE"] = "float16"
    print(f"⚡ Hardware: NVIDIA GPU detected ({torch.cuda.get_device_name(0)}) — CUDA acceleration enabled.")
else:
    os.environ["WHISPER_DEVICE"] = "cpu"
    os.environ["WHISPER_COMPUTE"] = "int8"
    print("ℹ️ Hardware: Running on CPU mode (int8). For 5x faster processing, switch runtime to T4 GPU.")

# 1. System packages & hardware tools
print("📦 Step 1/5: Installing system dependencies (ffmpeg, fonts)...")
subprocess.run(
    "apt-get update -qq && apt-get install -y -qq ffmpeg libsm6 libxext6 libgl1 fonts-freefont-ttf fonts-dejavu-core",
    shell=True, check=True
)

# 2. Download Cloudflared for instant zero-config public HTTPS tunnel
print("🌐 Step 2/5: Setting up Cloudflare tunnel binary...")
if not os.path.exists("/content/cloudflared"):
    subprocess.run(
        "wget -q -nc -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared",
        shell=True, check=True
    )

# 3. Clone or update repository from GitHub
REPO_URL = "https://github.com/shreeharsh-patil/PeakCut.git"
REPO_DIR = "/content/PeakCut"

print(f"📥 Step 3/5: Fetching repository from {REPO_URL}...")
if not os.path.exists(REPO_DIR):
    subprocess.run(f"git clone {REPO_URL} {REPO_DIR}", shell=True, check=True)
else:
    subprocess.run(f"git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} reset --hard origin/main", shell=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 4. Install Python dependencies
print("🐍 Step 4/5: Installing Python dependencies...")
subprocess.run("pip install -q -r requirements.txt", shell=True)

# 5. Start Cloudflare Tunnel with file-based logging (avoids OS pipe deadlock)

# Google Drive Mount (if enabled)
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        print("📁 Mounting Google Drive...")
        drive.mount('/content/drive')
        drive_dir = '/content/drive/MyDrive/PeakCut_Shorts'
        os.makedirs(drive_dir, exist_ok=True)
        os.environ['OUTPUT_DIR'] = drive_dir
        print(f"✅ Google Drive connected! Output clips will be saved in: {drive_dir}")
    except Exception as e:
        print(f"⚠️ Could not mount Google Drive ({e}). Falling back to local output directory.")
print("🚀 Step 5/5: Starting Cloudflare tunnel & PeakCut Studio server...")
log_path = "/content/cloudflared.log"
if os.path.exists(log_path):
    os.remove(log_path)

tunnel_cmd = f"/content/cloudflared tunnel --url http://127.0.0.1:8000 --logfile {log_path}"
tunnel = subprocess.Popen(tunnel_cmd, shell=True)

# 6. Extract and display the public HTTPS link
public_url = None
for _ in range(60):
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
            if match:
                public_url = match.group(0)
                break
    time.sleep(0.5)

print("\n" + "=" * 70)
if public_url:
    print(f"🎉 PEAKCUT STUDIO IS LIVE AT: {public_url}")
    print("👉 Click the link above to open PeakCut Studio in your browser!")
else:
    print("⚠️ Cloudflare tunnel starting... check /content/cloudflared.log for details.")
print("=" * 70 + "\n")

# Rich HTML launch banner
try:
    from IPython.display import display, HTML
    if public_url:
        mode_note = "🟢 Running in background mode! Cell completed." if RUN_IN_BACKGROUND else "⚠️ <b>Keep this cell running (spinning icon is normal)</b> to keep PeakCut Studio active."
        display(HTML(f'''
        <div style="background: linear-gradient(135deg, #18181b, #09090b); border: 2px solid #8b5cf6; border-radius: 12px; padding: 22px; text-align: center; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
          <h2 style="color: #ffffff; margin: 0 0 8px 0; font-size: 22px;">🎉 PeakCut Studio is Live!</h2>
          <p style="color: #c4b5fd; font-size: 14px; margin-bottom: 18px;">{mode_note}</p>
          <a href="{public_url}" target="_blank" style="background: linear-gradient(135deg, #7c3aed, #6d28d9); color: white; padding: 12px 30px; text-decoration: none; border-radius: 8px; font-weight: 700; font-size: 16px; display: inline-block; box-shadow: 0 4px 15px rgba(124, 58, 237, 0.4);">
            🚀 Open PeakCut Studio ↗
          </a>
          <p style="color: #71717a; font-size: 12px; margin-top: 14px;">
            Public Link: <a href="{public_url}" target="_blank" style="color: #a78bfa; text-decoration: underline;">{public_url}</a>
          </p>
        </div>
        '''))
except Exception:
    pass

# 7. Start FastAPI Server with PeakCut Studio UI
start_cmd = "python -m uvicorn app:app --host 0.0.0.0 --port 8000" if os.path.exists("app.py") else "python -m studio.server"
if RUN_IN_BACKGROUND:
    print("🟢 PeakCut Studio is running in background mode! Use the link above.")
    subprocess.Popen(f"{start_cmd} > /content/uvicorn.log 2>&1", shell=True)
else:
    print("🟢 PeakCut Studio server is active!")
    print("ℹ️ NOTE: This cell will show a spinning icon — this is NORMAL and keeps the server alive!")
    print("ℹ️ Do NOT click the Stop button unless you want to shut down PeakCut Studio.\n")
    try:
        subprocess.run(start_cmd, shell=True)
    except KeyboardInterrupt:
        print("\n🛑 PeakCut Studio server stopped by user.")


In [ ]:
# @title 📋 Server Status & Live Logs (Optional)
# Run this cell to check whether the server is running or inspect recent logs
import subprocess
print("🔍 Active Server Processes:")
!ps aux | grep -E "uvicorn|cloudflared" | grep -v grep || echo "No running server found."

print("\n📜 Recent Logs:")
!tail -n 25 /content/uvicorn.log 2>/dev/null || echo "(Running in foreground mode in cell above)"


### ✂️ Alternative: Process Directly from Command Line (Headless Mode)
You can also process videos directly in Colab without the Web UI using this cell:

In [ ]:
# @title 🎬 Process Video via CLI (Direct)
VIDEO_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ" # @param {type:"string"}
FORMAT = "vertical" # @param ["vertical", "horizontal", "square", "auto"]

import os
if os.path.exists("/content/PeakCut"):
    os.chdir("/content/PeakCut")

!python main.py --url "$VIDEO_URL" --format "$FORMAT" --output "/content/output"


### 🦙 Zero-API Mode: Local Gemma / Vision Model on GPU (Saved in Google Drive)
Run Google's open **Gemma / PaliGemma** model directly on Colab's free NVIDIA GPU. The weights are stored in your **Google Drive** (`MyDrive/models/ollama`), so it only downloads **once** on day 1. Every future session loads in seconds from Drive with **zero API keys and zero rate limits**!

In [ ]:
# @title 🦙 Start Local Gemma Model on GPU (Zero-Key Mode)
# @markdown Choose which open model to load onto the Colab GPU:
MODEL_NAME = "gemma2:2b" # @param ["gemma2:2b", "gemma2:9b", "paligemma"]
SAVE_MODEL_TO_DRIVE = True # @param {type:"boolean"}

import os, sys, subprocess, time

# 1. Mount Google Drive for persistent model storage
if SAVE_MODEL_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    model_dir = "/content/drive/MyDrive/models/ollama"
    os.makedirs(model_dir, exist_ok=True)
    os.environ["OLLAMA_MODELS"] = model_dir
    print(f"📁 Persistent model storage configured in Google Drive: {model_dir}")

# 2. Install Ollama local inference engine if not present
print("📦 Installing local GPU inference engine...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

# 3. Start background server
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# 4. Pull/load model (reads from Drive if already downloaded, else downloads once)
print(f"🚀 Loading {MODEL_NAME} on NVIDIA GPU...")
subprocess.run(["ollama", "pull", MODEL_NAME], check=True)

# 5. Wire PeakCut / OpenShorts to use local GPU model
os.environ["LLM_PROVIDER"] = "ollama"
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_MODEL"] = MODEL_NAME
print("\n" + "="*65)
print(f"✅ Local {MODEL_NAME} active at http://localhost:11434/v1!")
print("🎉 All video processing will now run 100% locally with ZERO API keys and ZERO rate limits!")
print("="*65)


### 🤖 Automated Batch Video Generation (Hands-Free / Google Drive Sync)
Process multiple YouTube videos automatically in sequence. All generated vertical clips are saved directly into your **Google Drive** (`MyDrive/PeakCut_Shorts`), persisting even after the Colab session ends.

In [ ]:
# @title 🤖 Batch Autonomous Shorts Generator (Google Drive Sync)
# @markdown Paste multiple YouTube URLs (one per line). Colab will automatically download each video,
# @markdown transcribe on NVIDIA GPU, cut viral vertical 9:16 shorts, and save them straight to Google Drive!
YOUTUBE_URLS = """https://www.youtube.com/watch?v=kYfNvmF0Bqw
https://www.youtube.com/watch?v=dQw4w9WgXcQ""" # @param {type:"string"}
CLIPS_PER_VIDEO = 3 # @param {type:"integer"}
FORMAT = "vertical" # @param ["vertical", "horizontal", "square", "auto"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}

import os, sys, subprocess
if os.path.exists("/content/PeakCut"):
    os.chdir("/content/PeakCut")

output_dir = "/content/output"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        output_dir = "/content/drive/MyDrive/PeakCut_Shorts"
        os.makedirs(output_dir, exist_ok=True)
        print(f"📁 Google Drive mounted! Saving to: {output_dir}")
    except Exception as e:
        print(f"⚠️ Could not mount Drive ({e}). Falling back to local /content/output.")

urls = [u.strip() for u in YOUTUBE_URLS.strip().splitlines() if u.strip() and not u.strip().startswith("#")]
print(f"🎬 Queued {len(urls)} video(s) for automatic batch processing.\n" + "="*60)

urls_arg = ",".join(urls)
cmd = [sys.executable, "batch_runner.py", "--urls", urls_arg, "--output", output_dir, "--target-clips", str(CLIPS_PER_VIDEO), "--format", FORMAT]
subprocess.run(cmd)


In [ ]:
# @title 📤 Auto-Publish Clip to YouTube Shorts (Optional)
# @markdown Upload any generated clip directly to your YouTube Channel as a YouTube Short.
VIDEO_PATH = "" # @param {type:"string"}
TITLE = "Viral Highlight #Shorts" # @param {type:"string"} 
DESCRIPTION = "Created with PeakCut Studio #Shorts #Viral" # @param {type:"string"}
PRIVACY_STATUS = "unlisted" # @param ["unlisted", "public", "private"]
YOUTUBE_ACCESS_TOKEN = "" # @param {type:"string"}

import os, sys, subprocess
if YOUTUBE_ACCESS_TOKEN.strip():
    os.environ["YOUTUBE_ACCESS_TOKEN"] = YOUTUBE_ACCESS_TOKEN.strip()

if not VIDEO_PATH.strip():
    print("ℹ️ Enter the path to a video clip (e.g. /content/drive/MyDrive/PeakCut_Shorts/... or output/...)")
else:
    cmd = [sys.executable, "youtube_uploader.py", "--video", VIDEO_PATH.strip(), "--title", TITLE, "--description", DESCRIPTION, "--privacy", PRIVACY_STATUS]
    subprocess.run(cmd)
